# Which detector should I build?

**Author:** Antoine ([@antowan](https://github.com/antowan))

**You can read the results here without running any code or connecting a station.** These are measurements from an Orin NX 16 GB, not predictions for other devices.

| Model | mAP 50:95, 80 classes (published) | mAP 50:95, car / person / truck | FP16 ms per image, batch 1 | FP16 ms per image, batch 8 |
|---|---:|---:|---:|---:|
| YOLO26n | 39.92 (40.1) | 40.67 | 5.3 | 3.4 |
| YOLOv10n | 38.34 (38.5) | 40.54 | 5.5 | 3.8 |
| YOLOv7-tiny | 37.13 (n/a) | 39.41 | 8.4 | 4.7 |

For these three classes, YOLO26n and YOLOv10n are only **0.13 mAP points** apart. YOLOv7-tiny is **1.26 points** behind YOLO26n. YOLO26n also had the lowest raw inference time here, which makes it a good [first build](03-build-yolo26.ipynb). Check difficult examples from your own cameras before choosing on accuracy alone.

## 1. How well does it detect?

A detector must name the object **and** put its box in the right place. Intersection over union (IoU) measures how much the predicted and labelled boxes overlap. **mAP 50:95** averages precision across classes and ten IoU thresholds, from 0.50 to 0.95. A score of 40 is not “40% of objects detected.”

Use **80 classes** to compare with published COCO results. On all 5,000 COCO val2017 images at 640 px, the raw FP16 engines scored within 0.2 points of the [YOLO26n end-to-end reference (40.1) and YOLOv10n paper reference (38.5)](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.md#accuracy). We found no directly comparable official 640 px result for these YOLOv7-tiny weights. The **three-class** column scores the same predictions for car, person and truck only; it is not a published benchmark or a test of the deployed callback.

## 2. Explore the recorded measurements (optional, local Jupyter)

Run the next cells in order to see per-class accuracy, timing by batch size and the difference between FP32 and FP16. They read the [saved measurement files](../doc/benchmarks/orin-nx-16gb-jp6.0/) and require no GPU. On GitHub, the same [detailed tables](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.md) are already rendered.

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert Path("tools/compare_models.py").is_file(), "Open this notebook from the repository (notebooks/ folder)"

sys.path.insert(0, "tools")
import compare_models

# Point this to tmp/benchmarks to look at your own measurements (final section).
RESULTS = Path("doc/benchmarks/orin-nx-16gb-jp6.0")
entries = compare_models.collect(RESULTS)
print("Measured:", ", ".join(entry["flavor"]["display_name"] for entry in entries))
print(compare_models.environment(entries))

### Accuracy by class

Read across a row to see whether a similar overall score hides a difference on a class you care about. These model evaluations use labelled images rather than tracked video.

In [ ]:
display(Markdown(compare_models.accuracy_table(entries)))

### How long does inference take?

The first table divides median batch time by the number of images in the batch; the second gives time for the **whole request**. A batch of eight is not eight cameras running live. Timing used GPU-resident synthetic inputs and excludes decoding, tracking and video outputs. For YOLOv7-tiny it also excludes the deployment engine’s EfficientNMS plugin. The idle timing run used rebuilt engines; the accuracy results are from an earlier run. See the [measurement conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).

In [ ]:
display(Markdown(compare_models.latency_table(entries)))
display(Markdown("**All runtimes and precisions** (median of one request, in milliseconds):"))
display(Markdown(compare_models.precision_table(entries)))

### See the trade-off

Lower latency is better (left); higher mAP is better (right).

In [ ]:
chart = RESULTS / "comparison.png"
if chart.is_file():
    display(Image(filename=str(chart)))
else:
    print("No chart in this results folder; use the tables above.")

### Does FP16 change detection quality?

Compare FP32 and FP16 scores at the **same batch size**. The difference is in mAP points, not percent speed-up.

In [ ]:
display(Markdown(compare_models.parity_table(entries)))

## 3. How many live cameras does that mean?

Don’t divide model throughput by 30. A camera frame must also be decoded, tracked and, if requested, encoded for output. In separate **720p30 H.264 RTSP tests**, the measured boundaries were **7 cameras for YOLO26n, 6 for YOLOv10n and 4 for YOLOv7-tiny**. Each input had to average at least **29 processed FPS** over 120 seconds, not deliver every frame at 30 FPS.

Those counts held for the three output profiles tested across all models. They do not cover every setting, long runs or connected WebRTC viewers. On this development build, **single-input YOLO26n WebRTC + OSD and continuous-recording tests failed** even though higher counts passed. Use the [capacity report](../doc/benchmarks/orin-nx-16gb-jp6.0/capacity.md) for the actual settings, repeats and output diagnostics.

## 4. Repeat the model measurements (optional, station only)

Leave `MEASURE = False` to skip **all** downloads, installs and benchmarks below. To make new measurements, use App-Builder on an idle development station with internet access and room for COCO val2017 and the engines. Reserve the GPU rather than stopping someone else’s pipeline. Allow roughly 40–60 minutes per flavor plus the download.

The cells export models, build raw engines, time inference and score labelled images. Results go to `tmp/benchmarks`, not the reference directory. Use `LIMIT = 0` for a full COCO comparison; a subset is only a smoke test. These commands do not run the live camera-capacity tests.

In [ ]:
MEASURE = False          # set to True to run the cells below
LIMIT = 0                # score only the first LIMIT images (0 = all 5,000); use e.g. 200 for a quick check
FLAVORS = ["yolov7", "yolov10", "yolo26"]

os.environ.update(MEASURE=str(int(MEASURE)), LIMIT=str(LIMIT), FLAVORS=" ".join(FLAVORS))

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
mkdir -p tmp/coco && cd tmp/coco
[ -d val2017 ] || { curl -fsSLO http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip && rm val2017.zip; }
[ -f annotations/instances_val2017.json ] || { curl -fsSLO http://images.cocodataset.org/annotations/annotations_trainval2017.zip \
    && unzip -q -o annotations_trainval2017.zip annotations/instances_val2017.json && rm annotations_trainval2017.zip; }
python3 -m pip install --user -r ../../tools/requirements-accuracy.txt
python3 -m pip install --user --no-deps -r ../../tools/requirements-export.txt
ls val2017 | wc -l

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/export_model.py "$flavor"
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/measure.py "$flavor" --annotations tmp/coco/annotations/instances_val2017.json \
        --images tmp/coco/val2017 --limit "$LIMIT" --results tmp/benchmarks
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
python3 tools/compare_models.py tmp/benchmarks

Open `tmp/benchmarks/comparison.md` for your report. To display your own tables here, change `RESULTS` in section 2 and rerun **only section 2**. The summary at the top and the camera counts in section 3 always describe the committed reference runs.